# 🚀 Fine-Tuning XLM-RoBERTa for Pakistani NER (v3 Retraining Package)
**Competition:** TechFest NER Challenge (20 Pakistani Entity Classes)
**Architecture:** `xlm-roberta-base`
**Data Version:** `v3` (`v3/train.jsonl` - 7,997 sentences, `v3/dev.jsonl` - 887 sentences)

> ⚠️ **CRITICAL WARNING ABOUT NESTED CLONE PATHS:**
> In Google Colab, executing duplicate git clone commands can create a nested subfolder:
> `/content/AI-Odyssey/ner-pk/AI-Odyssey/ner-pk`
> Always execute the directory check cell below to ensure `os.getcwd()` is `/content/AI-Odyssey/ner-pk`.

In [ ]:
# 1. GPU Verification and Google Drive Mount
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone Repository or Navigate with Nested Path Protection
import os

target_repo_dir = "/content/AI-Odyssey/ner-pk"
if not os.path.exists(target_repo_dir):
    %cd /content
    !git clone https://github.com/Muhammad-Hassan06/AI-Odyssey.git

# Set working directory safely
os.chdir(target_repo_dir)
print("Current Working Directory:", os.getcwd())
assert os.path.exists("v3/train.jsonl"), f"Path error! v3/train.jsonl not found in {os.getcwd()}"

In [ ]:
# 3. Check Library Versions (NO library downgrades) & Data File SHA256 Hashes
import hashlib
import torch
import transformers
import seqeval
from pathlib import Path

print(f"PyTorch:      {torch.__version__} (CUDA: {torch.cuda.is_available()})")
print(f"Transformers: {transformers.__version__}")
print(f"Seqeval:      {seqeval.__version__}")

def get_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while chunk := f.read(65536):
            h.update(chunk)
    return h.hexdigest()

print("\n--- Data File Hashes ---")
for split in ["v3/train.jsonl", "v3/dev.jsonl", "v3/benchmark.jsonl"]:
    p = Path(split)
    assert p.exists(), f"Missing {split}!"
    print(f"{split:<22}: {get_sha256(p)}")

In [ ]:
# 4. Validate Schema Prior to Training
# Strictly asserts 20 allowed classes, valid BIO tags, and no corrupted spans
!python scripts/validate_schema.py v3

In [ ]:
# 5. Execute Fine-Tuning (8 Epochs, lr 2.5e-5, Batch 16, fp16, Class-Weighted Loss)
# Checkpoint chosen strictly by best seqeval micro-F1 on DEV
!python src/train.py \
    --model_name xlm-roberta-base \
    --data_dir v3 \
    --output_dir /content/drive/MyDrive/ner_pk_models_v3/best_model \
    --epochs 8 \
    --lr 2.5e-5 \
    --batch_size 16 \
    --max_length 256 \
    --fp16 \
    --use_class_weights \
    --seed 42

In [ ]:
# 6. Checkpoint Integrity Verification
import os, json

model_dir = "/content/drive/MyDrive/ner_pk_models_v3/best_model"
assert os.path.exists(os.path.join(model_dir, "model.safetensors")), "model.safetensors missing!"
assert os.path.exists(os.path.join(model_dir, "config.json")), "config.json missing!"

with open(os.path.join(model_dir, "config.json")) as f:
    cfg = json.load(f)
assert len(cfg["id2label"]) == 41, f"Expected 41 labels, got {len(cfg['id2label'])}"
print("✔ Checkpoint strictly verified: 41 BIO labels matching 20 competition classes!")

In [ ]:
# 7. Zip Checkpoint for Rapid Local Download
!zip -r /content/best_ner_model_v3.zip /content/drive/MyDrive/ner_pk_models_v3/best_model
print("Archive created: /content/best_ner_model_v3.zip")